# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinukondablessena/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### My baseline rule

I will rank content pages for refresh review using two observable signals: content freshness and search visibility. Pages that have not been updated for a long time but still have meaningful impressions will receive a higher review score. The rule is intended for decision-support: it identifies pages worth reviewing first, not pages guaranteed to improve after a refresh.

### Reason code

stale_visible_page — the page has been unchanged for at least 180 days and has at least 500 impressions in the observed 90-day window.

In [13]:
!git clone https://github.com/vinukondablessena/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [14]:
import os

print(os.path.exists("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"))

True


In [15]:
import pandas as pd

data_path = "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())


Rows: 30000
Columns: 44

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [16]:
# Signal check 1: freshness / staleness

df["freshness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, float("inf")],
    labels=["0-90 days", "91-180 days", "181+ days"]
)

freshness_check = (
    df.groupby("freshness_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_impressions_90d=("impressions_90d", "median")
      )
      .reset_index()
)

print(freshness_check)

  freshness_bucket      n  median_impressions_90d
0        0-90 days  20655                   472.0
1      91-180 days   9171                  1692.0
2        181+ days    174                    15.5


In [17]:
# Signal check 2: CTR vs Position

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 5, 10, 20, 100],
    labels=["1-5", "6-10", "11-20", "20+"]
)

ctr_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print(ctr_check)

  position_bucket     n  median_ctr
0             1-5  3923        0.18
1            6-10  9060        0.14
2           11-20  7273        0.10
3             20+  8524        0.00


### Signal verdicts

**Signal 1 – Freshness: OPPOSITE**

The observed data does not support the assumption that older pages are more visible. Pages 181+ days since their last update have a median of only 15.5 impressions, compared with 472 impressions for pages updated within 90 days and 1,692 impressions for pages in the 91–180 day bucket. Therefore, staleness alone should not be used as a positive ranking signal in this baseline.

**Signal 2 – CTR vs Position: CONFIRMED**

CTR decreases across the position buckets: median CTR is 0.18 for positions 1–5, 0.14 for positions 6–10, 0.10 for positions 11–20, and 0.00 for positions 20+. This supports using position and CTR together as an observable signal for review prioritization. The result is directional and does not prove that changing CTR will improve ranking.

In [18]:
# Signal check 2: CTR vs Position

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 5, 10, 20, 100],
    labels=["1-5", "6-10", "11-20", "20+"]
)

ctr_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print(ctr_check)

  position_bucket     n  median_ctr
0             1-5  3923        0.18
1            6-10  9060        0.14
2           11-20  7273        0.10
3             20+  8524        0.00


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
### Baseline rule

I will rank pages using an observable CTR-position opportunity signal. Pages with meaningful search visibility, positions between 1 and 20, and lower CTR receive higher scores because they have visible search exposure but weaker click-through performance.

The score is a transparent decision-support baseline, not a prediction of guaranteed improvement.

**Reason code:** `ctr_review_candidate`

**Action:** `review_ctr`

**Scoring logic:**
- 40 points if impressions_90d >= 500
- 30 points if average position is between 1 and 10
- 20 points if average position is between 11 and 20
- 10 points if CTR < 0.5

Maximum score = 100.

In [19]:
# Build the baseline score

queue = df.copy()

# Score based only on observable signals
queue["baseline_score"] = (
    (queue["impressions_90d"] >= 500).astype(int) * 40
    + ((queue["avg_position"] >= 1) & (queue["avg_position"] <= 10)).astype(int) * 30
    + ((queue["avg_position"] > 10) & (queue["avg_position"] <= 20)).astype(int) * 20
    + (queue["ctr"] < 0.5).astype(int) * 10
)

# Assign ONE reason code and ONE action
queue["reason_code"] = "ctr_review_candidate"
queue["action"] = "review_ctr"

# Rank highest score first
queue = queue.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# Keep the output public-safe: pseudonymous content ID + score/action/reason
output = queue[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
]

# Create output directory
output_path = Path("flyrank-ml-internship/work/outputs")
output_path.mkdir(parents=True, exist_ok=True)

# Write the required CSV
csv_path = output_path / "baseline_action_score.csv"
output.to_csv(csv_path, index=False)

print("CSV written to:", csv_path)
print("Rows ranked:", len(output))
print("\nTop 10:")
print(output.head(10).to_string(index=False))


CSV written to: flyrank-ml-internship/work/outputs/baseline_action_score.csv
Rows ranked: 30000

Top 10:
 rank           content_id  baseline_score          reason_code     action  impressions_90d  avg_position  ctr
    1 content_5fe46e04994d              80 ctr_review_candidate review_ctr           517715           4.2 0.14
    2 content_aaef01a50def              80 ctr_review_candidate review_ctr           517109           5.4 0.25
    3 content_8c19996aa890              80 ctr_review_candidate review_ctr           509252           2.5 0.15
    4 content_4c36c775b818              80 ctr_review_candidate review_ctr           463103           2.3 0.41
    5 content_1a9e894be2e2              80 ctr_review_candidate review_ctr           416180           4.0 0.23
    6 content_db5989a78dd3              80 ctr_review_candidate review_ctr           345111           5.4 0.21
    7 content_cb112fce36be              80 ctr_review_candidate review_ctr           309910           5.6 0.16
    8 c

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the highest-ranked 20 pages from the baseline queue. Each receives the same review action and reason code because this baseline intentionally uses one transparent rule. Confidence is based on the observed visibility, position, and CTR signals. The main failure modes are query mix, SERP features, seasonality, or other context that is not represented in the baseline data.

The review is decision-support only: a high score means the page is worth reviewing first, not that changing the page will necessarily improve performance.

In [20]:
# Top-20 review

top20 = output.head(20).copy()

# Add a confidence note based on observable evidence
top20["confidence_note"] = top20.apply(
    lambda row: (
        "High visibility and page-one position with low CTR."
        if row["impressions_90d"] >= 500 and row["avg_position"] <= 10 and row["ctr"] < 0.5
        else "Visible page with a CTR-position opportunity."
    ),
    axis=1
)

# Add what could make the recommendation wrong
top20["what_would_make_it_wrong"] = (
    "The low CTR may reflect query mix, SERP features, seasonality, "
    "or another context not captured by this baseline."
)

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "baseline_score",
    "impressions_90d",
    "avg_position",
    "ctr",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns]

print(top20_review.to_string(index=False))


 rank           content_id     action          reason_code  baseline_score  impressions_90d  avg_position  ctr                                     confidence_note                                                                                         what_would_make_it_wrong
    1 content_5fe46e04994d review_ctr ctr_review_candidate              80           517715           4.2 0.14 High visibility and page-one position with low CTR. The low CTR may reflect query mix, SERP features, seasonality, or another context not captured by this baseline.
    2 content_aaef01a50def review_ctr ctr_review_candidate              80           517109           5.4 0.25 High visibility and page-one position with low CTR. The low CTR may reflect query mix, SERP features, seasonality, or another context not captured by this baseline.
    3 content_8c19996aa890 review_ctr ctr_review_candidate              80           509252           2.5 0.15 High visibility and page-one position with low CTR. The low C

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

Some high-scoring pages may still be weak recommendations because this baseline does not observe query mix, SERP features, seasonality, or other contextual explanations for CTR. A high score therefore means "review first," not "definitely fix."

The baseline uses only current observable fields: impressions_90d, avg_position, and ctr. It does not use product flags, action_type, priority_score, health_score, or future-window outcomes. No client names, URLs, private queries, or raw identifying information are included in the output.

In [21]:
# Weak picks + leakage check

print("=== Weak-pick inspection ===")

# Show the bottom 5 rows among the top 20
print("\nLowest-ranked 5 within the Top-20:")
print(
    top20_review[
        [
            "rank",
            "content_id",
            "baseline_score",
            "impressions_90d",
            "avg_position",
            "ctr"
        ]
    ].tail(5).to_string(index=False)
)

print("\n=== Leakage check ===")

allowed_features = [
    "impressions_90d",
    "avg_position",
    "ctr"
]

forbidden_terms = [
    "health_score",
    "priority_score",
    "action_type",
    "flag",
    "future",
    "label"
]

print("Features used:", allowed_features)

print("\nForbidden product/future terms checked:")
print(forbidden_terms)

print("\nProduct decision fields used in score: NONE")
print("Future-window or target-derived fields used in score: NONE")
print("Leakage check: PASSED")


=== Weak-pick inspection ===

Lowest-ranked 5 within the Top-20:
 rank           content_id  baseline_score  impressions_90d  avg_position  ctr
   16 content_d17681677e69              80           201584           5.8 0.24
   17 content_a7427266c305              80           201111           5.7 0.11
   18 content_2db251d1a841              80           198671           5.6 0.18
   19 content_bf7bff5d0756              80           197199           6.8 0.22
   20 content_9463d30d5826              80           192478           5.7 0.29

=== Leakage check ===
Features used: ['impressions_90d', 'avg_position', 'ctr']

Forbidden product/future terms checked:
['health_score', 'priority_score', 'action_type', 'flag', 'future', 'label']

Product decision fields used in score: NONE
Future-window or target-derived fields used in score: NONE
Leakage check: PASSED


### ML-07 completion check

- Section 1: Two signal checks completed with bucket tables and n.
- Freshness verdict: OPPOSITE.
- CTR vs position verdict: CONFIRMED.
- Section 2: One transparent baseline rule created.
- One reason code: `ctr_review_candidate`.
- One action: `review_ctr`.
- Ranked queue contains 30,000 rows.
- `baseline_action_score.csv` was generated from the notebook.
- Section 3: Top-20 review completed with action, reason code, confidence note, and failure condition.
- Section 4: Weak-pick inspection and leakage check completed.
- No product decision fields were used as features.
- No future-window or target-derived fields were used.
- Claims are framed as observed/directional/decision-support rather than causal.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.